# Импорты

In [1]:
import plotly.express as px
import numpy as np

# Входные данные

Уравнение:<br>
$
\begin{cases} \displaystyle
    u'' - \frac{3}{(x + 1)^2} u = \frac{-1,5}{\sqrt{x + 1}} \\
    3u(0) - u'(0) = 1\\
    u'(1) = \sqrt{2}
\end{cases}
$

Решение:<br>
$\displaystyle
u(x) = \frac{2}{3} (x + 1)^{\frac{3}{2}}
$

In [2]:
N = 100

def p(x: float) -> float:
    return 0

def q(x: float) -> float:
    return -3 / ((x + 1) ** 2)

def f(x: float) -> float:
    return -1.5 / ((x + 1) ** 0.5)

def u(x: float) -> float:
    return 2 / 3 * (x + 1) ** (3 / 2)

alpha_1 = 3
beta_1 = -1
gamma_1 = 1

alpha_2 = 0
beta_2 = 1
gamma_2 = np.sqrt(2)

h = 1 / N
x_real = np.linspace(0, 1, N + 1)
x_real = np.concatenate([[x_real[0] - 2 * h, x_real[0] - h], x_real, [x_real[-1] + h, x_real[-1] + 2 * h]])
def x_i(i: int) -> float:
    if i >= len(x_real):
        return x_real[-1] + (i - (len(x_real) - 1)) * h
    if i < 0:
        return x_real[0] + i * h
    return x_real[i]

# Решение

## Метод монотонной прогонки

In [3]:
def monotonous_reduction(A: np.array, B: np.array, C: np.array, F: np.array) -> np.array:
    n = len(A) - 1

    alpha = np.zeros(n + 1)
    beta = np.zeros(n + 1)
    U = np.zeros(n + 1)

    alpha[1] = -C[0] / B[0]
    beta[1] = F[0] / B[0]

    for i in range(1, n):
        alpha[i + 1] = -C[i] / (B[i] + alpha[i] * A[i])
        beta[i + 1] = (F[i] - beta[i] * A[i]) / (B[i] + alpha[i] * A[i])

    U[n] = (F[n] - beta[n] * A[n]) / (B[n] + alpha[n] * A[n])

    for i in range(n - 1, -1, -1):
        U[i] = alpha[i + 1] * U[i + 1] + beta[i + 1]

    return U

## Определение кубического B-сплайна

B-сплайн степени $p$ определяется по формуле Кокса-де Бура: <br>
$\displaystyle
B_{i, 0}(x) = \begin{cases}
    1, & x \in [x_i; x_{i + 1}) \\
    0, & x \notin [x_i; x_{i + 1})
\end{cases} \\
B_{i, p}(x) = \frac{x - x_i}{x_{i + p} - x_i} B_{i, p - 1}(x) + \frac{x_{i + p + 1} - x}{x_{i + p + 1} - x_{i + 1}} B_{i + 1, p - 1}(x)
$

Тогда, совершая вычисления, получим следующие значения: <br>
$\displaystyle
B_{i, 1}(x) = \frac{x - x_i}{x_{i + 1} - x_i} B_{i, 0}(x) + \frac{x_{i + 2} - x}{x_{i + 2} - x_{i + 1}} B_{i + 1, 0}(x) = \begin{cases}
    \displaystyle \frac{x - x_i}{x_{i + 1} - x_i}, & x \in [x_i; x_{i + 1}) \\
    \displaystyle \frac{x_{i + 2} - x}{x_{i + 2} - x_{i + 1}}, & x \in [x_{i + 1}; x_{i + 2}) \\
    0, & x \notin [x_i; x_{i + 2})
\end{cases} \\
B_{i, 2}(x) = \frac{x - x_i}{x_{i + 2} - x_i} B_{i, 1}(x) + \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 1}} B_{i + 1, 1}(x) = \begin{cases}
    \displaystyle \frac{x - x_i}{x_{i + 1} - x_i} \cdot \frac{x - x_i}{x_{i + 2} - x_i}, & x \in [x_i; x_{i + 1}) \\
    \displaystyle \frac{x_{i + 2} - x}{x_{i + 2} - x_{i + 1}} \cdot \frac{x - x_i}{x_{i + 2} - x_i} + \frac{x - x_{i + 1}}{x_{i + 2} - x_{i + 1}} \cdot \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 1}}, & x \in [x_{i + 1}; x_{i + 2}) \\
    \displaystyle \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 2}} \cdot \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 1}}, & x \in [x_{i + 2}; x_{i + 3}) \\
    0, & x \notin [x_i; x_{i + 3})
\end{cases} \\
B_{i, 3}(x) = \frac{x - x_i}{x_{i + 3} - x_i} B_{i, 2}(x) + \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 1}} B_{i + 1, 2}(x) = \begin{cases}
    \displaystyle \frac{x - x_i}{x_{i + 1} - x_i} \cdot \frac{x - x_i}{x_{i + 2} - x_i} \cdot \frac{x - x_i}{x_{i + 3} - x_i}, & x \in [x_i; x_{i + 1}) \\
    \displaystyle \frac{x_{i + 2} - x}{x_{i + 2} - x_{i + 1}} \cdot \frac{x - x_i}{x_{i + 2} - x_i} \cdot \frac{x - x_i}{x_{i + 3} - x_i} + \frac{x - x_{i + 1}}{x_{i + 2} - x_{i + 1}} \cdot \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 1}} \cdot \frac{x - x_i}{x_{i + 3} - x_i} + \frac{x - x_{i + 1}}{x_{i + 2} - x_{i + 1}} \cdot \frac{x - x_{i + 1}}{x_{i + 3} - x_{i + 1}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 1}}, & x \in [x_{i + 1}; x_{i + 2}) \\
    \displaystyle \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 2}} \cdot \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 1}} \cdot \frac{x - x_i}{x_{i + 3} - x_i} + \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 2}} \cdot \frac{x - x_{i + 1}}{x_{i + 3} - x_{i + 1}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 1}} + \frac{x - x_{i + 2}}{x_{i + 3} - x_{i + 2}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 2}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 1}}, & x \in [x_{i + 2}; x_{i + 3}) \\
    \displaystyle \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 3}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 2}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 1}}, & x \in [x_{i + 3}; x_{i + 4}) \\
    0, & x \notin [x_i; x_{i + 4})
\end{cases}
$

Мы получили кубический B-сплайн для узлов с индексацией вида: $x_0, x_1, \ldots, x_{n + 4}$: <br>
$
B_{i, 3}(x) = \begin{cases}
    \displaystyle \frac{x - x_i}{x_{i + 1} - x_i} \cdot \frac{x - x_i}{x_{i + 2} - x_i} \cdot \frac{x - x_i}{x_{i + 3} - x_i}, & x \in [x_i; x_{i + 1}) \\
    \displaystyle \frac{x_{i + 2} - x}{x_{i + 2} - x_{i + 1}} \cdot \frac{x - x_i}{x_{i + 2} - x_i} \cdot \frac{x - x_i}{x_{i + 3} - x_i} + \frac{x - x_{i + 1}}{x_{i + 2} - x_{i + 1}} \cdot \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 1}} \cdot \frac{x - x_i}{x_{i + 3} - x_i} + \frac{x - x_{i + 1}}{x_{i + 2} - x_{i + 1}} \cdot \frac{x - x_{i + 1}}{x_{i + 3} - x_{i + 1}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 1}}, & x \in [x_{i + 1}; x_{i + 2}) \\
    \displaystyle \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 2}} \cdot \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 1}} \cdot \frac{x - x_i}{x_{i + 3} - x_i} + \frac{x_{i + 3} - x}{x_{i + 3} - x_{i + 2}} \cdot \frac{x - x_{i + 1}}{x_{i + 3} - x_{i + 1}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 1}} + \frac{x - x_{i + 2}}{x_{i + 3} - x_{i + 2}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 2}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 1}}, & x \in [x_{i + 2}; x_{i + 3}) \\
    \displaystyle \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 3}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 2}} \cdot \frac{x_{i + 4} - x}{x_{i + 4} - x_{i + 1}}, & x \in [x_{i + 3}; x_{i + 4}) \\
    0, & x \notin [x_i; x_{i + 4})
\end{cases}
$

In [4]:
def B_i(x: float, i: int) -> float:
    if x < x_i(i):
        return 0
    elif x_i(i) <= x < x_i(i + 1):
        return (
            (x - x_i(i)) / (x_i(i + 1) - x_i(i)) *
            (x - x_i(i)) / (x_i(i + 2) - x_i(i)) * 
            (x - x_i(i)) / (x_i(i + 3) - x_i(i))
        )
    elif x_i(i + 1) <= x < x_i(i + 2):
        return (
            (x_i(i + 2) - x) / (x_i(i + 2) - x_i(i + 1)) *
            (x - x_i(i)) / (x_i(i + 2) - x_i(i)) * 
            (x - x_i(i)) / (x_i(i + 3) - x_i(i))
            +
            (x - x_i(i + 1)) / (x_i(i + 2) - x_i(i + 1)) *
            (x_i(i + 3) - x) / (x_i(i + 3) - x_i(i + 1)) *
            (x - x_i(i)) / (x_i(i + 3) - x_i(i))
            +
            (x - x_i(i + 1)) / (x_i(i + 2) - x_i(i + 1)) *
            (x - x_i(i + 1)) / (x_i(i + 3) - x_i(i + 1)) *
            (x_i(i + 4) - x) / (x_i(i + 4) - x_i(i + 1))
        )
    elif x_i(i + 2) <= x < x_i(i + 3):
        return (
            (x_i(i + 3) - x) / (x_i(i + 3) - x_i(i + 2)) *
            (x_i(i + 3) - x) / (x_i(i + 3) - x_i(i + 1)) *
            (x - x_i(i)) / (x_i(i + 3) - x_i(i))
            +
            (x_i(i + 3) - x) / (x_i(i + 3) - x_i(i + 2)) *
            (x - x_i(i + 1)) / (x_i(i + 3) - x_i(i + 1)) *
            (x_i(i + 4) - x) / (x_i(i + 4) - x_i(i + 1))
            +
            (x - x_i(i + 2)) / (x_i(i + 3) - x_i(i + 2)) *
            (x_i(i + 4) - x) / (x_i(i + 4) - x_i(i + 2)) *
            (x_i(i + 4) - x) / (x_i(i + 4) - x_i(i + 1))
        )
    elif x_i(i + 3) <= x < x_i(i + 4):
        return (
            (x_i(i + 4) - x) / (x_i(i + 4) - x_i(i + 3)) *
            (x_i(i + 4) - x) / (x_i(i + 4) - x_i(i + 2)) *
            (x_i(i + 4) - x) / (x_i(i + 4) - x_i(i + 1))
        )
    elif x >= x_i(i + 4):
        return 0

## Построение решения

Решение строится как: <br>
$\displaystyle 
S(x) = \sum_{i = -1}^{n + 1} b_i B_i(x) \\
B_i(x) = B_{i - 2, 3}(x) \\
h_{-j} = h_{j - 1}, ~~ h_{n - 1 + j} = h_{n - j}, ~~ j = 1, 2, 3 \\
$

Для поиска коэффициентов $b_i$ строится система уравнений: <br>
$\displaystyle
b_{i - 1} A_i + b_i C_i + b_{i + 1} D_i = F_i, ~~ i = \overline{0, n} \\
A_i = \frac{1}{x_{i + 1} - x_{i - 2}} \left( 1 - \frac{1}{2} p_i h_i + \frac{1}{6} q_i h_i^2 \right) = \frac{1}{x_{i + 1} - x_{i - 2}} \left( 1 - \frac{1}{2} \frac{h_i^2}{(x_i + 1)^2} \right) = \frac{1}{x_{i + 1} - x_{i - 2}} \left( 1 - \frac{1}{2} \frac{(x_{i + 1} - x_i)^2}{(x_i + 1)^2} \right) \\
D_i = \frac{1}{x_{i + 2} - x_{i - 1}} \left( 1 + \frac{1}{2} p_i h_{i - 1} + \frac{1}{6} q_i h_{i - 1}^2 \right) = \frac{1}{x_{i + 2} - x_{i - 1}} \left( 1 - \frac{1}{2} \frac{h_{i - 1}^2}{(x_i + 1)^2} \right) = \frac{1}{x_{i + 2} - x_{i - 1}} \left( 1 - \frac{1}{2} \frac{(x_i - x_{i - 1})^2}{(x_i + 1)^2} \right) \\
C_i = -A_i - D_i + \frac{1}{6} q_i (h_i + h_{i - 1}) = -\frac{1}{x_{i + 1} - x_{i - 2}} \left( 1 - \frac{1}{2} \frac{h_i^2}{(x_i + 1)^2} \right) - \frac{1}{x_{i + 2} - x_{i - 1}} \left( 1 - \frac{1}{2} \frac{h_{i - 1}^2}{(x_i + 1)^2} \right) - \frac{1}{2} \frac{h_i + h_{i - 1}}{(x_i + 1)^2} = -\frac{1}{x_{i + 1} - x_{i - 2}} \left( 1 - \frac{1}{2} \frac{(x_{i + 1} - x_i)^2}{(x_i + 1)^2} \right) - \frac{1}{x_{i + 2} - x_{i - 1}} \left( 1 - \frac{1}{2} \frac{(x_i - x_{i - 1})^2}{(x_i + 1)^2} \right) - \frac{1}{2} \frac{x_{i + 1} - x_{i - 1}}{(x_i + 1)^2} \\
F_i = \frac{1}{6} f_i (h_i + h_{i - 1}) = -\frac{1}{4} \frac{h_i + h_{i - 1}}{\sqrt{x_i + 1}} = -\frac{1}{4} \frac{x_{i + 1} - x_{i - 1}}{\sqrt{x_i + 1}}
$

Для $b_{-1}$: <br>
$\displaystyle
b_{-1} A_{- 1} + b_0 C_{-1} + b_1 D_{-1} = F_{-1} \\
A_{-1} = \alpha_1 h_0 - 3 \beta_1 = 3h_0 + 3 = 3(x_1 - x_0 + 1) = 3(x_1 + 1) \\
C_{-1} = 2 \alpha_1 (h_0 + h_1) = 6(h_0 + h_1) = 6(x_2 - x_0) = 6x_2 \\
D_{-1} = \alpha_1 h_0 + 3 \beta_1 = 3h_0 - 3 = 3(x_1 - x_0 - 1) = 3(x_1 - 1) \\
F_{-1} = 2 \gamma_1 (2h_0 + h_1) = 2(x_2 + x_1 - 2x_0) = 2(x_2 + x_1) \\
b_{-1} = \frac{F_{-1} - b_0 C_{-1} - b_1 D_{-1}}{A_{- 1}} = \frac{2(x_2 + x_1) - 6 b_0 x_2 - 3 b_1 (x_1 - 1)}{3(x_1 + 1)}
$

Для $b_{n + 1}$: <br>
$\displaystyle
b_{n - 1} A_{n + 1} + b_n C_{n + 1} + b_{n + 1} D_{n + 1} = F_{n + 1} \\
A_{n + 1} = \alpha_2 h_{n - 1} - 3 \beta_2 = -3 \\
C_{n + 1} = 2 \alpha_2 (h_{n - 2} + h_{n + 1}) = 0 \\
D_{n + 1} = \alpha_2 h_{n - 1} + 3\beta_2 = 3 \\
F_{n + 1} = 2\gamma_2 (2h_{n - 1} + h_{n - 2}) = 2\sqrt{2} (2h_{n - 1} + h_{n - 2}) = 2\sqrt{2} (2x_n - x_{n - 1} - x_{n - 2}) = 2\sqrt{2} (2 - x_{n - 1} - x_{n - 2}) \\
b_{n + 1} = \frac{F_{n + 1} - b_n C_{n + 1} - b_{n - 1} A_{n + 1}}{D_{n + 1}} = \frac{2\sqrt{2} (2 - x_{n - 1} - x_{n - 2}) + 3 b_{n - 1}}{3}
$

Тогда итоговая система: <br>
$\displaystyle
b_0 \left( C_0 - \frac{C_{-1} A_0}{A_{-1}} \right) + b_1 \left( D_0 - \frac{D_{-1} A_0}{A_{-1}} \right) = \left( F_0 - \frac{F_{-1} A_0}{A_{-1}} \right) \\
b_{i - 1} A_i + b_i C_i + b_{i + 1} D_i = F_i, ~~ i = \overline{1, n - 1} \\
b_{n - 1} \left( A_n - \frac{A_{n + 1} D_n}{D_{n + 1}} \right) + b_n \left( C_n - \frac{C_{n + 1} D_n}{D_{n + 1}} \right) = \left( F_n - \frac{F_{n + 1} D_n}{D_{n + 1}} \right) \\
\left(\begin{array}{cccccccc}
    \displaystyle C_0 - \frac{C_{-1} A_0}{A_{-1}} & \displaystyle D_0 - \frac{D_{-1} A_0}{A_{-1}} & 0 & 0 & \ldots & 0 & 0 & 0 & 0 \\
    A_1 & C_1 & D_1 & 0 & \ldots & 0 & 0 & 0 & 0 \\
    0 & A_2 & C_2 & D_2 & \ldots & 0 & 0 & 0 & 0 \\
    \vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots & \vdots & \vdots \\
    0 & 0 & 0 & 0 & \ldots & A_{n - 2} & C_{n - 2} & D_{n - 2} & 0 \\
    0 & 0 & 0 & 0 & \ldots & 0 & A_{n - 1} & C_{n - 1} & D_{n - 1} \\
    0 & 0 & 0 & 0 & \ldots & 0 & 0 & \displaystyle A_n - \frac{A_{n + 1} D_n}{D_{n + 1}} & \displaystyle C_n - \frac{C_{n + 1} D_n}{D_{n + 1}}
\end{array}\right) \cdot \left(\begin{array}{c}
    b_0 \\ b_1 \\ b_2 \\ \vdots \\ b_{n - 2} \\ b_{n - 1} \\ b_n
\end{array}\right) = \left(\begin{array}{c}
    \displaystyle F_0 - \frac{F_{-1} A_0}{A_{-1}} \\ F_1 \\ F_2 \\ \vdots \\ F_{n - 2} \\ F_{n - 1} \\ \displaystyle F_n - \frac{F_{n + 1} D_n}{D_{n + 1}}
\end{array}\right)
$

In [5]:
A = np.array([
    1 / (x_i(i + 1) - x_i(i - 2)) * (1 - 0.5 * p(x_i(i)) * (x_i(i + 1) - x_i(i)) + 1 / 6 * q(x_i(i)) * (x_i(i + 1) - x_i(i)) ** 2) for i in range(N + 1)
])
D = np.array([
    1 / (x_i(i + 2) - x_i(i - 1)) * (1 + 0.5 * p(x_i(i)) * (x_i(i) - x_i(i - 1)) + 1 / 6 * q(x_i(i)) * (x_i(i) - x_i(i - 1)) ** 2) for i in range(N + 1)
])
C = np.array([
    -A[i] - D[i] + 1 / 6 * q(x_i(i)) * (x_i(i + 1) - x_i(i - 1)) for i in range(N + 1)
])
F = np.array([
    1 / 6 * f(x_i(i)) * (x_i(i + 1) - x_i(i - 1)) for i in range(N + 1)
])

A_minus_1 = alpha_1 * (x_i(1) - x_i(0)) - 3 * beta_1
A_N_plus_1 = alpha_2 * (x_i(N) - x_i(N - 1)) - 3 * beta_2
D_minus_1 = alpha_1 * (x_i(1) - x_i(0)) + 3 * beta_1
D_N_plus_1 = alpha_2 * (x_i(N) - x_i(N - 1)) + 3 * beta_2
C_minus_1 = 2 * alpha_1 * (x_i(2) - x_i(0))
C_N_plus_1 = 2 * alpha_2 * (x_i(N + 2) - x_i(N - 2))
F_minus_1 = 2 * gamma_1 * (x_i(2) + x_i(1) - 2 * x_i(0))
F_N_plus_1 = 2 * gamma_2 * (2 * x_i(N) - x_i(N - 1) - x_i(N - 2))

C[0] = C[0] - (C_minus_1 * A[0]) / A_minus_1
C[N] = C[N] - (C_N_plus_1 * D[N]) / D_N_plus_1
F[0] = F[0] - (F_minus_1 * A[0]) / A_minus_1
F[N] = F[N] - (F_N_plus_1 * D[N]) / D_N_plus_1
A[N] = A[N] - (A_N_plus_1 * D[N]) / D_N_plus_1
D[0] = D[0] - (D_minus_1 * A[0]) / A_minus_1
A[0] = 0
D[N] = 0

b = monotonous_reduction(A, C, D, F)
b = np.concatenate([
    [(F_minus_1 - b[0] * C_minus_1 - b[1] * D_minus_1) / A_minus_1],
    b,
    [(F_N_plus_1 - b[N] * C_N_plus_1 - b[N - 1] * A_N_plus_1) / D_N_plus_1]
])

def S(x: float) -> float:
    return sum(b[i] * B_i(x, i - 1) for i in range(N + 3))

# Визуализация

In [6]:
x_points = np.linspace(0, 1, 1000)
y_i = np.array([u(x) for x in x_points])
S_i = np.array([S(x) for x in x_points])

## Сравнение

In [7]:
figure = px.line(x=x_points, y=y_i, markers=False)
figure.update_traces(name="Точное решение", showlegend=True)

figure.add_scatter(x=x_points, y=S_i, name="Численное решение")

figure.show()

## Погрешность

In [8]:
figure = px.line(x=x_points, y=np.abs(y_i - S_i), markers=False)
figure.update_layout(yaxis=dict(exponentformat="power"))
figure.show()